# 第 6 章 · 多智能体模式与生态对比（收官章）

> 本章目标：
> 1. 用 LangGraph 手工实现经典的 **Supervisor（监管者）多智能体模式**——与 ADK 第 4 章正面交锋；
> 2. 概览 LangSmith 观测与 LangGraph Platform 部署；
> 3. 送上**两大框架终极大对照表**与全教程总结。

---

## 1. 多智能体的四种经典模式

```mermaid
flowchart TB
    subgraph P1["① Supervisor 监管者"]
        S1["🧭 Supervisor<br/>中央调度"] --> W1A["Worker A"]
        S1 --> W1B["Worker B"]
        W1A --> S1
        W1B --> S1
    end
    subgraph P2["② Handoff 移交"]
        H1["Agent A"] -->|"Command(goto=...)"| H2["Agent B"]
        H2 -->|"移交"| H3["Agent C"]
    end
    subgraph P3["③ Hierarchical 层级"]
        M1["总监管"] --> M2["子团队主管1"]
        M1 --> M3["子团队主管2"]
        M2 --> M4["组员"]
    end
    subgraph P4["④ Pipeline 流水线"]
        L1["A"] --> L2["B"] --> L3["C"]
    end
    style S1 fill:#e8f0fe,stroke:#4285f4,stroke-width:2px
```

| 模式 | 特点 | ADK 对应 |
|---|---|---|
| Supervisor | 中央大脑统一调度，Worker 干完汇报 | LLM 委派（root + sub_agents） |
| Handoff | Agent 之间直接"交棒"，无中心 | `transfer_to_agent` |
| Hierarchical | Supervisor 套 Supervisor | 嵌套 sub_agents |
| Pipeline | 固定顺序流水线 | `SequentialAgent` |

> 📌 社区的 `langgraph-supervisor` 与 `langgraph-swarm` 包把前两种模式做成了开箱组件。但**手工实现一遍**是理解它们的最佳方式——下面动手。

---

## 2. 实战：手工实现 Supervisor 模式

**场景**：内容小组——`supervisor` 指挥 `researcher`（调研员，带搜索工具）与 `writer`（写手）协作完成任务。

```mermaid
flowchart LR
    S((START)) --> SUP["🧭 supervisor<br/>决定下一步"]
    SUP -->|"researcher"| R["🔍 researcher<br/>create_agent 子图<br/>（带搜索工具）"]
    SUP -->|"writer"| W["✍️ writer<br/>create_agent 子图"]
    R -->|"交回成果"| SUP
    W -->|"交回成稿"| SUP
    SUP -->|"FINISH"| E((END))
    style SUP fill:#e8f0fe,stroke:#4285f4,stroke-width:2px
```

设计要点：

1. **State**：`MessagesState` + 一个 `next` 字段（supervisor 的路由决定）；
2. **supervisor 节点**：用结构化输出让 LLM 只做选择题（`researcher` / `writer` / `FINISH`）——比让它自由发挥可靠得多；
3. **worker 节点**：直接复用第 3 章的 `create_agent`（每个 worker 内部都是一个完整的 ReAct 循环！）——**图嵌套图**；
4. **回边**：worker 干完回到 supervisor，直到 supervisor 说 FINISH。


In [1]:
import os
assert os.environ.get("DEEPSEEK_API_KEY"), "请先设置环境变量 DEEPSEEK_API_KEY"

from typing import Literal, TypedDict, Annotated
from pydantic import BaseModel
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain.agents import create_agent   # 注：旧写法 langgraph.prebuilt.create_react_agent 已迁移至此

llm = ChatOpenAI(model="deepseek-chat", api_key=os.environ["DEEPSEEK_API_KEY"],
                 base_url="https://api.deepseek.com", temperature=0)

# ---------- ① 定义两个 Worker（各自是完整的 ReAct Agent） ----------
@tool
def search_knowledge(topic: str) -> dict:
    """搜索内部知识库，获取某主题的事实要点。

    Args:
        topic: 要调研的主题。
    """
    kb = {"氢能源": "氢能源汽车：加氢3分钟续航600km；当前瓶颈是制氢成本与加氢站密度。",
          "固态电池": "固态电池：能量密度高、安全性好；量产难点在界面阻抗与成本。"}
    return {"facts": kb.get(topic, f"知识库中关于「{topic}」的资料有限")}

researcher_agent = create_agent(model=llm, tools=[search_knowledge])
writer_agent = create_agent(model=llm, tools=[])

# ---------- ② 定义团队 State ----------
class TeamState(TypedDict):
    messages: Annotated[list, add_messages]
    next: str

# ---------- ③ supervisor：结构化输出做路由 ----------
class RouteDecision(BaseModel):
    next: Literal["researcher", "writer", "FINISH"]
    reason: str

router_llm = llm.with_structured_output(RouteDecision, method="function_calling")  # DeepSeek 走 function calling 通道

def supervisor(state: TeamState) -> dict:
    history = "\n".join(f"[{m.type}] {m.content[:100]}" for m in state["messages"])
    decision = router_llm.invoke([
        ("system", "你是团队主管。规则：需要先调研就先派 researcher；资料够了派 writer 成稿；writer 已交稿则 FINISH。每步只派一个人。"),
        ("human", f"当前进展：\n{history}\n\n原始任务：{state['messages'][0].content}\n\n下一步派谁？"),
    ])
    print(f"  🧭 supervisor 决策 → {decision.next}（{decision.reason}）")
    return {"next": decision.next}

def researcher(state: TeamState) -> dict:
    r = researcher_agent.invoke({"messages": [HumanMessage(
        content=f"调研任务相关的信息，调用搜索工具，输出要点：{state['messages'][0].content}")]})
    return {"messages": [AIMessage(content="【调研结果】" + r["messages"][-1].content, name="researcher")]}

def writer(state: TeamState) -> dict:
    r = writer_agent.invoke({"messages": state["messages"] + [HumanMessage(
        content="基于以上调研结果，写一段 100 字左右的科普短文。")]})
    return {"messages": [AIMessage(content="【成稿】" + r["messages"][-1].content, name="writer")]}

# ---------- ④ 建图 ----------
b = StateGraph(TeamState)
b.add_node("supervisor", supervisor)
b.add_node("researcher", researcher)
b.add_node("writer", writer)
b.add_edge(START, "supervisor")
b.add_conditional_edges("supervisor", lambda s: s["next"],
                        {"researcher": "researcher", "writer": "writer", "FINISH": END})
b.add_edge("researcher", "supervisor")   # 回边：干完活向主管汇报
b.add_edge("writer", "supervisor")
team = b.compile()

r = team.invoke({"messages": [HumanMessage(content="调研氢能源汽车现状，并写一段科普短文")],
                 "next": ""})
print("═" * 60)
print("🏁 最终成果：\n", r["messages"][-1].content)


  🧭 supervisor 决策 → researcher（任务需要先调研氢能源汽车现状，目前还没有相关资料，需要先派研究员进行调研。）


  🧭 supervisor 决策 → writer（调研已完成，虽然知识库资料有限，但已有调研结果汇总，可以派writer基于现有资料撰写科普短文。）


  🧭 supervisor 决策 → FINISH（调研已完成，科普短文已成稿，任务可以结束。）
════════════════════════════════════════════════════════════
🏁 最终成果：
 【成稿】氢能源汽车以氢燃料电池为核心，将氢气与氧气反应产生的电能驱动车辆，排放物仅为水，是真正的“零排放”出行方案。它补能快、续航长，尤其适合重卡、公交等长途重载场景。尽管当前面临成本高、加氢站少等挑战，但随着绿氢技术成熟，氢能汽车有望与纯电车互补，共筑清洁未来。


> 🔍 观察 supervisor 的决策序列——通常是 `researcher → writer → FINISH`，但它**完全由 LLM 根据对话历史自主决定**。对比第 4 章的手工 ReAct：这里每个 worker 内部又是一个完整的 ReAct 循环——**LangGraph 的"图即节点"嵌套能力是层级多智能体的基石**。

> 💡 想要"移交"而非"汇报"式协作（Handoff），让工具返回 `Command(goto="另一个agent")` 即可；`langgraph-supervisor` 包则把本章模式封装成了一行 `create_supervisor(...)`。

---

## 3. 观测与评估：LangSmith

LangSmith 是 LangChain 生态的观测平台（SaaS）。接入只需环境变量，**代码零改动**：

```bash
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=ls__...
LANGSMITH_PROJECT=my-agent
```

之后每一次 `invoke` 都会留下完整 Trace：

```mermaid
flowchart LR
    T["Trace 树"] --> R1["run: team<br/>⏱️ 12.3s 🎫 8,214 tok"]
    R1 --> R2["run: supervisor<br/>LLM 决策详情"]
    R1 --> R3["run: researcher<br/>├── LLM 调用<br/>├── tool: search_knowledge<br/>└── LLM 总结"]
    R1 --> R4["run: writer ..."]
    style T fill:#fef7e0,stroke:#fbbc04,stroke-width:2px
```

| 能力 | 说明 | ADK 对应 |
|---|---|---|
| Tracing | 每次 LLM/工具调用的输入输出、耗时、token | `adk web` Events 面板 / OTel |
| Datasets & Evaluation | 建测试集、定义评分器、回归测试 | `adk eval` 评估集 |
| Prompt Playground | 在线调试提示词 | — |
| 监控告警 | 生产流量质量看板 | Cloud Monitoring |

> 📌 教学提示：没有 LangSmith 账号也不影响本教程学习；本地开发可用 `graph.get_graph().draw_mermaid()` + `stream_mode="updates"` 完成大部分调试。

---

## 4. 部署：LangGraph Platform

```mermaid
flowchart LR
    G["你的 graph.py"] --> CLI["langgraph CLI"]
    CLI --> DEV["langgraph dev<br/>本地开发服务器 + Studio 调试 UI"]
    CLI --> CLOUD["LangGraph Platform<br/>托管：API/定时任务/水平扩缩"]
    CLI --> SELF["自托管容器<br/>docker 镜像"]
```

| 选项 | 一句话 | ADK 对应 |
|---|---|---|
| `langgraph dev` + Studio | 本地起服务，浏览器里可视化调试图 | `adk web` |
| LangGraph Platform | 全托管生产运行时 | Vertex AI Agent Engine |
| 自托管 | 标准 Docker 部署 | `adk api_server` / Cloud Run |

---

## 5. 终极大对照：ADK vs LangChain/LangGraph

### 5.1 概念映射表（收藏级）

| 概念 | Google ADK | LangChain / LangGraph |
|---|---|---|
| 智能体 | `Agent(instruction, model, tools)` | `create_agent(model, tools, system_prompt)` |
| 模型抽象 | `BaseLlm`（Gemini / LiteLlm） | `BaseChatModel`（partner 包） |
| 工具 | 裸函数 / `FunctionTool` / `AgentTool` | `@tool` / `ToolNode` |
| 会话单元 | Session（events + state） | Thread（checkpoint 序列） |
| 工作记忆 | `session.state`（含 `user:` 前缀） | State（TypedDict + Reducer） |
| 长期记忆 | `MemoryService` | `Store` |
| 流水线 | `SequentialAgent` | 直线边链 / LCEL |
| 并行 | `ParallelAgent` | 扇出边 + 汇聚节点 |
| 循环 | `LoopAgent` + escalate | 条件边回边 |
| LLM 路由 | `sub_agents` + transfer | 条件边 / Supervisor / `Command(goto)` |
| 护栏 | Callbacks（6 种钩子） | Middleware（before/after/wrap） |
| 人机协同 | LongRunningFunctionTool | `interrupt` + `Command(resume)` |
| 时间旅行 | ❌ | ✅ `get_state_history` + `update_state` |
| 本地调试 UI | `adk web` | LangGraph Studio |
| 云端观测 | Cloud Trace / OTel | LangSmith |
| 托管部署 | Agent Engine / Cloud Run | LangGraph Platform |
| 数据流编排 | ❌（无此层） | ✅ LCEL（独有） |

### 5.2 一句话选型

```mermaid
flowchart TD
    Q{"你的场景？"}
    Q -->|"多智能体系统，要快速搭起来调"| A1["ADK：声明式积木，开发体验顺滑"]
    Q -->|"复杂控制流：中断/回放/分叉/审批"| A2["LangGraph：图的掌控力无出其右"]
    Q -->|"大量数据集成/RAG 管道"| A3["LangChain：集成生态最庞大"]
    Q -->|"GCP 全家桶用户"| A1
    Q -->|"需要最成熟的 Trace/评估 SaaS"| A2
    style A1 fill:#e8f0fe,stroke:#4285f4,stroke-width:2px
    style A2 fill:#e6f4ea,stroke:#34a853,stroke-width:2px
```

> 💡 **最后的忠告**：框架会过时，概念不会。工具调用、状态管理、ReAct 循环、人机协同、评估驱动开发——这五个概念在任何框架（包括明年出现的新框架）里都通用。本教程双线对照的设计，正是为了让你看到**同一概念在不同抽象下的投影**。

---

## 6. 全教程地图回顾

```mermaid
flowchart LR
    subgraph ADK["Google ADK 线路"]
        A1["01 快速上手"] --> A2["02 Agent与模型"] --> A3["03 工具"]
        A3 --> A4["04 多智能体"] --> A5["05 记忆体系"] --> A6["06 生产化"]
    end
    subgraph LC["LangChain / LangGraph 线路"]
        L1["01 生态与模型"] --> L2["02 LCEL"] --> L3["03 create_agent"]
        L3 --> L4["04 图与状态"] --> L5["05 持久化/人机协同"] --> L6["06 多智能体/生态"]
    end
    A4 -.对照.-> L6
    A5 -.对照.-> L5
    A6 -.对照.-> L6
    A3 -.对照.-> L3
```

**进阶方向**：
1. 把本教程的示例换成你的真实业务场景重写一遍；
2. 给任意一章的 Agent 写**评估集**（ADK evalset / LangSmith Dataset）——这是从"会搭"到"会养"的分水岭；
3. 关注 MCP 协议——两大框架都在把工具生态向 MCP 收敛；
4. 阅读源码：`create_agent`（LangChain/LangGraph）与 `Runner`（ADK）的核心都不超过几百行，读完你会对框架彻底祛魅。

---

## 📌 本章要点回顾

- Supervisor = supervisor 路由节点 + worker（ReAct 子图）+ 回边；图嵌套图是层级多智能体的基础；
- LangSmith 零代码接入 Trace 与评估；LangGraph Platform 对应 ADK 的 Agent Engine；
- 概念映射表建议收藏——它是你在两个框架间自由穿梭的护照。

🎉 **教程完结**！回到 [00-框架全景与选型对比](../00-框架全景与选型对比.ipynb) 重温决策树，相信此刻的你已经有了自己的答案。


---

## 🧪 本章练习

### 1. 可终止的 Supervisor 团队（基础）

在本章 Supervisor 示例中加入一名事实核查 worker。Supervisor 必须根据共享消息决定 worker 顺序和 `FINISH`，同时限制最大交接次数。准备无需核查、需要核查和信息不足三类任务，输出完整路由序列并证明系统总能终止。

### 2. Supervisor 与 Handoff 对照实验（进阶）

用同一个客服场景分别实现“worker 向 Supervisor 汇报”和 `Command(goto=...)` 直接 Handoff。比较上下文归属、用户体验、路由可控性、token 消耗和错误恢复；设计一个需要两个专业域协作的请求，观察两种模式如何处理。

### 3. 多智能体评估与观测（工程）

建立至少 20 条任务集，在 LangSmith 或自建 Trace 中记录路由准确率、任务成功率、无效交接次数、总模型调用数、延迟和成本。定义单 Agent 基线，判断增加 worker 是否真的提升效果；如果没有，提出删减或合并 Agent 的方案。

### 4. 双框架毕业项目（综合）

选择“研究报告、售后客服、代码维护”之一，分别用 ADK 与 LangGraph 实现功能等价的最小系统。必须包含工具、状态/记忆、一次确定性流程、一次模型路由、人工审批、可观测记录和评估集。用同一批任务比较正确率、延迟、成本、代码复杂度与故障恢复，并写出最终选型 ADR。

### 5. 面向当前 Agent 趋势的架构评审（开放）

围绕 MCP/工具协议、上下文工程、持久化执行、小模型专业 Agent、Agent 间协作与安全治理，评审练习 4 的架构。回答：哪些能力应该交给模型，哪些应固化为图或工作流，哪些必须由外部基础设施保证？同时提出在需求增长十倍时仍可演进的模块边界。

### 6. 反思：什么时候不该用多智能体？（开放）

找出一个可以被“单 Agent + 好工具”或普通确定性程序替代的多智能体设计，从可靠性、延迟、成本、调试难度和组织边界解释简化理由。给出明确的升级触发条件，而不是为了展示技术而保留多 Agent。
